# distance

## prepare

In [21]:
import os
import platform

import pandas as pd
import numpy as np
from gensim.models import KeyedVectors
from sklearn.metrics.pairwise import cosine_distances, cosine_similarity
from scipy.spatial.distance import cosine
from scipy.spatial import distance

In [22]:
import warnings

# RuntimeWarning 무시
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [23]:
os_system = platform.system() # 맥북은 Darwin, 윈도우는 Windows

# 현재 프로젝트 폴더 위치 지정. os.getcwd()는 지금 코드 실행하는 현 위치를 출력해줍니다.
study1_dir = os.getcwd()
model_path = '\\..\\pretrained\\GoogleNews-vectors-negative300.bin' if os_system == 'Windows' else '/../pretrained/GoogleNews-vectors-negative300.bin'

# data/processed 폴더 위치 지정
processed_data_dir = study1_dir + ('\\data\\processed\\' if os_system == 'Windows' else '/data/processed/')

In [24]:
# word2vec model 로딩
word2vec_model = KeyedVectors.load_word2vec_format(study1_dir + model_path, binary=True)

In [25]:
# 테이블 읽어오기
tbl_data = pd.read_csv(processed_data_dir + 'data_after_preprocessing.csv', encoding='ISO-8859-1')

In [26]:
seed_words = ['key', 'money', 'friend']
target_words = ['money', 'friend']

n_respond_words = 30 # 하나의 시드당 30개의 단어 응답
n_subject = len(tbl_data) # 210
n_dim_of_vector = 300

## 벡터 구하기

In [27]:
for seed_word in seed_words: # key, money, friend
    word_columns = [seed_word + str(i) for i in range(1, n_respond_words+1)] # key1~30, money1~30, friend1~30

    for column in word_columns:
        # vector field 생성
        tbl_data[column + '_vec'] = np.empty(n_subject, dtype=object)

        # 피험자 한 명의 응답 단어들 벡터 처리
        for i_subject in range(n_subject):
            try:
                response_word = tbl_data.iloc[i_subject][column]
                if pd.isna(response_word) or len(response_word.strip()) == 0:# NaN, 값이 빈 칸 & 응답안해서 '', ' '로 저장된 경우 걸러내기
                    tbl_data[column + '_vec'][i_subject] = None
                    continue

                if isinstance(response_word, str):
                    response_word = response_word.split()
                    response_word = [response_word for response_word in response_word if response_word not in ['is', 'a','to','of','and']]
                    if len(response_word) == 0: # 앞에서 걸러져서 결과가 없으면, 넘어가기
                        tbl_data[column + '_vec'][i_subject] = None
                        continue

                    vec_word2vec = np.zeros((n_dim_of_vector, 0))  # 300차원의 빈 행렬 생성

                    for i_el in range(len(response_word)):
                        try:
                            vec_word2vec_in = word2vec_model[response_word[i_el]]
                        except:
                            vec_word2vec_in = word2vec_model[response_word[i_el].capitalize()]
                        # reshape: 벡터의 형태를 바꿔줄뿐. 300을 600 or 90으로 바꿀 순 없다.
                        vec_word2vec_in = vec_word2vec_in.reshape((n_dim_of_vector, 1))
                        vec_word2vec = np.hstack((vec_word2vec, vec_word2vec_in))  # 수평으로 벡터 쌓기
                    # 각 열(단어 벡터)에 대한 평균 계산
                    average_vector = np.mean(vec_word2vec, axis=1)
                    tbl_data[column + '_vec'][i_subject] = average_vector
            except:
                pass

/var/folders/99/w9lwt31s6gzbvts3vs5x6myr0000gn/T/ipykernel_17315/2264760989.py:35: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tbl_data[column + '_vec'][i_subject] = average_vector
/var/folders/99/w9lwt31s6gzbvts3vs5x6myr0000gn/T/ipykernel_17315/2264760989.py:35: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tbl_data[column + '_vec'][i_subject] = average_vector
/var/folders/99/w9lwt31s6gzbvts3vs5x6myr0000gn/T/ipykernel_17315/2264760989.py:35: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/us

# distances between target word

## distances between each response words and target word

90개의 응답 단어들 각각과 타겟 단어와의 거리를 구한다.

In [28]:
word_columns = [f'{seed_word}{i}' for seed_word in seed_words for i in range(1, n_respond_words + 1)]
# word_columns

In [29]:
# jaccard_seed_target, euclidean_seed_target 컬럼 미리 생성(빈 값)
for target_word in target_words:
    for seed_word in seed_words:
        word_columns = [seed_word + str(i) for i in range(1, n_respond_words + 1)]
        for column in word_columns: 
                column_name = f'jaccard_{column}_{target_word}'
                tbl_data = tbl_data.assign(**{column_name: None})

for target_word in target_words:
    for seed_word in seed_words:
        word_columns = [seed_word + str(i) for i in range(1, n_respond_words + 1)]
        for column in word_columns: 
                column_name = f'euclidean_{column}_{target_word}'
                tbl_data = tbl_data.assign(**{column_name: None})

tbl_data.columns

Index(['subject', 'key1', 'key2', 'key3', 'key4', 'key5', 'key6', 'key7',
       'key8', 'key9',
       ...
       'euclidean_friend21_friend', 'euclidean_friend22_friend',
       'euclidean_friend23_friend', 'euclidean_friend24_friend',
       'euclidean_friend25_friend', 'euclidean_friend26_friend',
       'euclidean_friend27_friend', 'euclidean_friend28_friend',
       'euclidean_friend29_friend', 'euclidean_friend30_friend'],
      dtype='object', length=541)

In [30]:
for i_subject in range(n_subject):
    # print('subject: ', i_subject)

    for target_word in target_words: # money, friend
        target_word_vec = word2vec_model[target_word]

        for seed_word in seed_words: # key, money, friend
            for i_word, column in enumerate(word_columns):
                try:
                    response_word_vec = tbl_data.iloc[i_subject][f'{column}_vec']
                    if response_word_vec is None:
                        continue
                    ## Calculate Jaccard Distance
                    # 두 벡터를 집합으로 변환
                    set_target = set(target_word_vec)
                    set_response = set(response_word_vec)

                    intersection = len(set_target.intersection(set_response))
                    union = len(set_target.union(set_response))
                    jaccard_distance = 1.0 - (intersection / union)
                    tbl_data.at[i_subject, f'jaccard_{seed_word}{i_word+1}_{target_word}'] = jaccard_distance

                    ## euclidean
                    euclidean_distance = distance.euclidean(target_word_vec, response_word_vec)
                    tbl_data.at[i_subject, f'euclidean_{seed_word}{i_word+1}_{target_word}'] = euclidean_distance


                except Exception as e:
                    print(f"An error occurred for subject {i_subject}: {str(e)}")
                    continue



In [31]:
tbl_data[0:3]

,subject,key1,key2,key3,key4,key5,key6,key7,key8,key9,...,euclidean_friend21_friend,euclidean_friend22_friend,euclidean_friend23_friend,euclidean_friend24_friend,euclidean_friend25_friend,euclidean_friend26_friend,euclidean_friend27_friend,euclidean_friend28_friend,euclidean_friend29_friend,euclidean_friend30_friend
0,1,card,bank,money,green,yellow,blue,eye,nose,smell,...,3.248842,3.329705,3.745543,None,3.976221,3.673209,4.271607,4.367179,3.823481,4.466066
1,2,lock,door,big,lion,roar,scary,ghost,dark,halloween,...,3.400866,3.615671,3.246348,3.419952,4.061933,3.667616,3.887027,3.78684,3.34577,3.394863
2,3,door,window,curtain,draft,cold,snow,scarf,wooly hat,winter,...,3.380811,4.185898,3.941965,4.157359,4.04215,3.779102,3.961814,4.16909,3.433853,3.956976


In [32]:
# vector 컬럼들 드롭 ( csv용량이 너무 커지는 것을 방지 )
drop_columns = tbl_data.columns[91:181]
tbl_data = tbl_data.drop(drop_columns, axis='columns')

In [33]:
# 단어 있는 버전 csv 저장
tbl_data.to_csv(processed_data_dir + 'jaccard_euclid_distance_with_words.csv', index=None)

In [34]:
# 단어 컬럼들 드롭
drop_columns = tbl_data.columns[1:91]
tbl_data = tbl_data.drop(drop_columns, axis='columns')

tbl_data.to_csv(processed_data_dir + 'jaccard_euclid_distance.csv', index=None)